# Lab 01 · Faça a reta aprender
**Machine Learning na Prática · Dia 1 · Módulo 02: Regressão Linear**

Os dados de alunos já chegam **limpos**, do jeito que você deixou no Lab 00. Agora o foco é o modelo:

| Etapa | O que fazer |
|---|---|
| 1. Olhar | carregar os dados prontos e visualizar horas × nota |
| 2. Ajustar | descida do gradiente com NumPy, acompanhar a MSE, comparar com `LinearRegression` |
| 3. Avaliar | prever alunos novos; MAE, MSE e R² no teste; regressão múltipla |
| 4. Taxa de aprendizado | convergência, lentidão e divergência |

Procure as células marcadas com **`# TODO`** e substitua os `___`.

> **Gabarito.** Versão do professor, com todas as células resolvidas. Não compartilhe antes da atividade.

### Bibliotecas
No **Google Colab** tudo já vem instalado: esta célula só confere. Em outro ambiente (Jupyter no seu PC), ela instala o que faltar.

In [ ]:
import importlib.util, subprocess, sys
pacotes = {'numpy': 'numpy', 'pandas': 'pandas', 'matplotlib': 'matplotlib', 'sklearn': 'scikit-learn'}
faltando = [pip for modulo, pip in pacotes.items() if importlib.util.find_spec(modulo) is None]
if faltando:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *faltando])
print('Bibliotecas prontas!' if not faltando else f'Instalado agora: {", ".join(faltando)}')

## 0 · Preparação (célula pronta)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

plt.rcParams['figure.figsize'] = (8, 4)

### Os dados limpos do Lab 00
A célula abaixo gera os mesmos dados e aplica a **mesma limpeza** do Lab 00 (duplicados, turno, outliers, alvo faltante).
Se você ainda tem o arquivo `alunos_limpo.csv` do Lab 00 nesta sessão, ele é usado direto.

In [ ]:
def gerar_dados(n=400, seed=42):
    rng = np.random.default_rng(seed)
    horas = np.clip(rng.normal(6, 2.2, n), 0.5, 14).round(1)
    faltas = rng.poisson(3, n)
    turno = rng.choice(['manhã', 'tarde', 'noite'], n, p=[.45, .3, .25])
    monitoria = rng.choice(['sim', 'não'], n, p=[.35, .65])
    idade = rng.integers(17, 31, n)
    renda = np.round(rng.lognormal(8.2, 0.5, n), -1)          # R$ por mês, escala grande
    nota = (2.0 + 0.45 * horas - 0.25 * faltas + 0.6 * (monitoria == 'sim')
            - 0.4 * (turno == 'noite') + rng.normal(0, 0.9, n))
    nota = np.clip(nota, 0, 10).round(1)
    df = pd.DataFrame({'horas_estudo': horas, 'faltas': faltas, 'turno': turno,
                       'monitoria': monitoria, 'idade': idade, 'renda_familiar': renda,
                       'nota_final': nota})
    df['situacao'] = np.where(df['nota_final'] >= 6, 'aprovado', 'reprovado')

    # ---- problemas do mundo real ----
    sujo = rng.choice(n, 40, replace=False)
    df.loc[sujo[:14], 'turno'] = rng.choice(['Manhã', 'MANHÃ ', 'manha', 'Noite', ' tarde'], 14)
    df.loc[rng.choice(n, 20, replace=False), 'horas_estudo'] = np.nan
    df.loc[rng.choice(n, 15, replace=False), 'faltas'] = np.nan
    df.loc[rng.choice(n, 8, replace=False), 'nota_final'] = np.nan
    outl = rng.choice(n, 4, replace=False)
    df.loc[outl, 'horas_estudo'] = df.loc[outl, 'horas_estudo'].fillna(7) * 10   # vírgula esquecida
    df = pd.concat([df, df.sample(12, random_state=seed)], ignore_index=True)   # duplicados
    return df.sample(frac=1, random_state=seed).reset_index(drop=True)

def limpar(df):
    """A mesma limpeza do Lab 00."""
    df = df.drop_duplicates().copy()
    df['turno'] = df['turno'].str.strip().str.lower().replace({'manha': 'manhã'})
    df.loc[df['horas_estudo'] > 24, 'horas_estudo'] = np.nan
    return df.dropna(subset=['nota_final']).reset_index(drop=True)

import os
df = pd.read_csv('alunos_limpo.csv') if os.path.exists('alunos_limpo.csv') else limpar(gerar_dados())
print(df.shape)
df.head()

> 🔎 **Gabarito · o que observar**
>
> - `(392, 8)`: os mesmos 392 alunos do fim do Lab 00, com o turno já padronizado e sem os 3 outliers gritantes.

## 1 · Uma feature, uma reta
Começamos com o caso mais simples: prever `nota_final` só com `horas_estudo`.
Para esta parte, deixamos de fora as linhas sem horas (nas próximas etapas o `Pipeline` cuida delas).

In [ ]:
d = df.dropna(subset=['horas_estudo'])
x_tr, x_te, y_tr, y_te = train_test_split(d['horas_estudo'].to_numpy(), d['nota_final'].to_numpy(),
                                          test_size=0.2, random_state=42)

# TODO: gráfico de dispersão horas × nota (treino)
plt.scatter(x_tr, y_tr, alpha=.5)
plt.xlabel('horas de estudo'); plt.ylabel('nota final'); plt.show()

> 🔎 **Gabarito · o que observar**
>
> - Sem os 21 faltantes em horas, sobram 371 alunos: **296 no treino e 75 no teste**.
> - A nuvem sobe da esquerda para a direita, mas é larga. Uma reta vai capturar a tendência, não cada aluno.
> - Repare no ponto isolado em **15 h com nota 0,4**, no canto inferior direito: é o erro de digitação (1,5 → 15) que a limpeza não pegou.

> 📋 **Gabarito · para o quadro**
>
> Antes de rodar a descida do gradiente, faça **uma conta à mão** com 4 alunos inventados (números redondos de propósito):
>
> | aluno | horas $x$ | nota $y$ |
> |---|---|---|
> | A | 2 | 3 |
> | B | 4 | 4 |
> | C | 6 | 4 |
> | D | 8 | 8 |
>
> **1. Uma reta qualquer:** $\hat y = 0{,}5\,x + 1{,}5$ → previsões **2,5 · 3,5 · 4,5 · 5,5**
>
> **2. O erro de cada aluno:** $y - \hat y$ → **0,5 · 0,5 · −0,5 · 2,5**
> (positivo: o modelo previu baixo; negativo: previu alto)
>
> **3. A MSE:** erros ao quadrado → 0,25 · 0,25 · 0,25 · **6,25**; soma 7; média **7 / 4 = 1,75**
> O aluno D sozinho responde por 6,25 dos 7 (quase 90%): **o quadrado faz o erro grande dominar.**
>
> **4. Um passo do gradiente**, começando do zero ($w = b = 0$, $\alpha = 0{,}01$):
> - com $w = b = 0$, o modelo prevê 0 para todos, então **erro = y** = 3 · 4 · 4 · 8
> - $x \cdot \text{erro}$ = 6 · 16 · 24 · 64 → média **27,5** → $\partial L/\partial w = -2 \cdot 27{,}5 =$ **−55**
> - média do erro = **4,75** → $\partial L/\partial b = -2 \cdot 4{,}75 =$ **−9,5**
> - $w = 0 - 0{,}01 \cdot (-55) =$ **0,55** e $b = 0 - 0{,}01 \cdot (-9{,}5) =$ **0,095**
> - com esse novo $w$ e $b$, a MSE cai de **26,25** para **≈ 4,70** num passo só.
>
> É exatamente o que a função abaixo faz, só que com 296 alunos e 3000 vezes. A célula a seguir confere as contas.

In [ ]:
# conferir as contas do quadro
# (nomes com q, de "quadro", para não sobrescrever as variáveis do lab)
xq = np.array([2, 4, 6, 8]); yq = np.array([3, 4, 4, 8])
eq = yq - (0.5 * xq + 1.5)
print('erro:', eq, '| MSE =', (eq ** 2).mean())

wq, bq, alpha = 0.0, 0.0, 0.01
eq = yq - (wq * xq + bq)
dw, db = -2 * (xq * eq).mean(), -2 * eq.mean()
wq, bq = wq - alpha * dw, bq - alpha * db
print(f'dw = {dw} | db = {db} → w = {wq:.3f} | b = {bq:.3f}')
print('MSE antes =', (yq ** 2).mean(), '| depois =', round(((yq - (wq * xq + bq)) ** 2).mean(), 2))

## 2 · Descida do gradiente com NumPy
O modelo é $\hat y = w\,x + b$ e a perda é a MSE: $L = \frac{1}{n}\sum (y - \hat y)^2$.
As derivadas são:

$$\frac{\partial L}{\partial w} = -\frac{2}{n}\sum x\,(y-\hat y) \qquad \frac{\partial L}{\partial b} = -\frac{2}{n}\sum (y-\hat y)$$

e a atualização é $w \leftarrow w - \alpha \, \frac{\partial L}{\partial w}$ (o mesmo para $b$).

In [ ]:
def descida_gradiente(x, y, alpha=0.02, epocas=3000):
    w, b = 0.0, 0.0
    historico = []
    for _ in range(epocas):
        # TODO: previsão da reta
        y_hat = w * x + b
        erro = y - y_hat
        historico.append((erro ** 2).mean())          # MSE desta época
        # TODO: gradientes da MSE
        dw = -2 * (x * erro).mean()
        db = -2 * erro.mean()
        # TODO: passo contra o gradiente
        w = w - alpha * dw
        b = b - alpha * db
    return w, b, historico

w, b, hist = descida_gradiente(x_tr, y_tr)
print(f'w = {w:.3f} | b = {b:.3f} | MSE final = {hist[-1]:.3f}')

plt.plot(hist); plt.yscale('log')
plt.xlabel('época'); plt.ylabel('MSE (treino)'); plt.title('A perda caindo'); plt.show()

> 🔎 **Gabarito · o que observar**
>
> - Resultado: **w ≈ 0,381** e **b ≈ 1,772**, com MSE final ≈ 1,235.
> - **w**: cada hora a mais de estudo soma cerca de 0,38 ponto na nota prevista.
> - **b**: a nota prevista para quem estuda 0 h. Ninguém nos dados estuda menos de 0,5 h, então é uma extrapolação.
> - A curva da MSE cai rápido no começo e depois fica **plana**: a partir daí, mais épocas não ensinam nada.

Agora compare com a solução exata do Scikit-learn (mínimos quadrados):

In [ ]:
# TODO: ajuste uma LinearRegression (ela espera X em 2D: x_tr.reshape(-1, 1))
reg = LinearRegression().fit(x_tr.reshape(-1, 1), y_tr)
print(f'Scikit-learn: w = {reg.coef_[0]:.3f} | b = {reg.intercept_:.3f}')
print(f'NumPy (GD):   w = {w:.3f} | b = {b:.3f}')

xs = np.linspace(0, 15, 100)
plt.scatter(x_tr, y_tr, alpha=.4)
plt.plot(xs, w * xs + b, 'C1', lw=3, label='descida do gradiente')
plt.plot(xs, reg.predict(xs.reshape(-1, 1)), 'k--', label='LinearRegression')
plt.legend(); plt.xlabel('horas'); plt.ylabel('nota'); plt.show()

> 🔎 **Gabarito · o que observar**
>
> - Os dois chegam ao **mesmo w e b** (0,381 e 1,772). A descida do gradiente achou, passo a passo, a mesma resposta que o Scikit-learn calcula de uma vez.
> - No gráfico, a reta laranja e a tracejada ficam uma em cima da outra.

**Curiosidade:** os dados foram gerados com uma regra escondida em que cada hora de estudo vale **+0,45** ponto.
O seu `w` chegou perto? Por que não é exatamente 0,45? (Dica: as outras colunas também mexem na nota.)

> 🔎 **Gabarito · o que observar**
>
> **Resposta esperada:** parte é ruído (a nota tem uma parcela aleatória). Mas a maior parte é **um único ponto**: o aluno em 15 h com nota 0,4.
> Ele está longe de todos no eixo x, e pontos assim têm muita “alavanca” sobre a reta. Tirando só ele, o `w` sobe de 0,381 para cerca de **0,43**.
> É o mesmo outlier que escapou da regra `> 24` no Lab 00: um erro de limpeza vira erro de modelo.

## 3 · Prever e avaliar
### 3.1 Alunos novos

In [ ]:
novos = np.array([2, 6, 10])
# TODO: use o modelo do Scikit-learn para prever as notas
for h, nota in zip(novos, reg.predict(novos.reshape(-1, 1))):
    print(f'{h:>2} h de estudo → nota prevista {nota:.2f}')

> 🔎 **Gabarito · o que observar**
>
> - 2 h → **2,53**; 6 h → **4,06**; 10 h → **5,59**.
> - A cada 4 horas a nota sobe 4 × 0,38 ≈ 1,53: a reta é linear.
> - Nem com 10 h o modelo prevê aprovação (≥ 6), porque horas sozinhas não bastam: faltas, turno e monitoria também mexem na nota.

### 3.2 Métricas no teste
- **MAE**: erro médio na unidade original (pontos de nota)
- **MSE**: média dos erros ao quadrado (pune erros grandes)
- **R²**: quanto melhor que sempre chutar a média (1 = perfeito, 0 = igual à média, pode ser negativo)

> 📋 **Gabarito · para o quadro**
>
> Com os **mesmos 4 alunos** do quadro anterior e a reta $\hat y = 0{,}5\,x + 1{,}5$ (erros **0,5 · 0,5 · −0,5 · 2,5**):
>
> | métrica | conta | resultado |
> |---|---|---|
> | **MAE** | (0,5 + 0,5 + 0,5 + 2,5) / 4 | **1,0 ponto** |
> | **MSE** | (0,25 + 0,25 + 0,25 + 6,25) / 4 | **1,75 ponto²** |
> | **R²** | 1 − 7 / 14,75 | **0,53** |
>
> **Como sai o R²:** a média das notas é $\bar y = 4{,}75$. Se o “modelo” chutasse sempre 4,75, os erros seriam −1,75 · −0,75 · −0,75 · 3,25, e a soma dos quadrados daria **14,75**. A reta erra **7** (soma dos quadrados). Então ela elimina 1 − 7/14,75 ≈ **53%** do erro de chutar a média.
>
> **O efeito do erro grande** (vale desenhar as duas linhas lado a lado):
>
> | | erros | MAE | MSE |
> |---|---|---|---|
> | D tirou 8 | 0,5 · 0,5 · −0,5 · **2,5** | 1,0 | 1,75 |
> | D tirou 6 | 0,5 · 0,5 · −0,5 · **0,5** | 0,5 | 0,25 |
>
> Um único erro 5× maior **dobra** o MAE e **multiplica a MSE por 7**. É por isso que a MSE é usada para treinar (empurra forte contra erros grandes) e o MAE para comunicar (“erra 1 ponto, em média”).

In [ ]:
# conferir as métricas do quadro
xq = np.array([2, 4, 6, 8]); yq_hat = 0.5 * xq + 1.5
for yq in (np.array([3, 4, 4, 8]), np.array([3, 4, 4, 6])):
    print(f'y = {yq} → MAE = {mean_absolute_error(yq, yq_hat):.2f} | MSE = {mean_squared_error(yq, yq_hat):.2f} | R² = {r2_score(yq, yq_hat):.2f}')

In [ ]:
pred = reg.predict(x_te.reshape(-1, 1))
# TODO: calcule as três métricas no TESTE
print(f'MAE = {mean_absolute_error(y_te, pred):.2f} ponto')
print(f'MSE = {mean_squared_error(y_te, pred):.2f}')
print(f'R²  = {r2_score(y_te, pred):.2f}')

> 🔎 **Gabarito · o que observar**
>
> - **MAE = 0,79**: em média, a previsão erra cerca de 0,8 ponto para cima ou para baixo.
> - **MSE = 0,97**: está em “pontos ao quadrado”, por isso é difícil de interpretar diretamente; serve para comparar modelos.
> - **R² = 0,47**: só com horas, o modelo explica quase metade da variação das notas no teste. O resto vem das outras features e do ruído.

### 3.3 Regressão múltipla: todas as features
Agora usamos **todas** as colunas, com o preparo do Lab 00 já montado (célula pronta).
O modelo passa a ser $\hat y = w_1 x_1 + w_2 x_2 + \dots + b$. Comparamos pela **validação cruzada no treino**; o teste fica para o fim.

In [ ]:
X = df.drop(columns=['nota_final', 'situacao'])
y = df['nota_final']
num, cat = ['horas_estudo', 'faltas', 'idade', 'renda_familiar'], ['turno', 'monitoria']
X_tr, X_te, Y_tr, Y_te = train_test_split(X, y, test_size=0.2, random_state=42)

preparo = ColumnTransformer([
    ('num', Pipeline([('imputar', SimpleImputer(strategy='median')), ('padronizar', StandardScaler())]), num),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat),
])
multipla = Pipeline([('preparo', preparo), ('regressao', LinearRegression())])

In [ ]:
# TODO: MAE da validação cruzada (5 folds) só no treino
# (o Scikit-learn devolve o MAE com sinal negativo — por isso o "-" na frente)
mae_cv = -cross_val_score(multipla, X_tr, Y_tr, cv=5, scoring='neg_mean_absolute_error')
print(f'MAE na validação cruzada: {mae_cv.mean():.2f} ± {mae_cv.std():.2f}')

# TODO: treine no treino inteiro e avalie UMA vez no teste
multipla.fit(X_tr, Y_tr)
pred = multipla.predict(X_te)
print(f'Teste → MAE = {mean_absolute_error(Y_te, pred):.2f} | R² = {r2_score(Y_te, pred):.2f}')

> 🔎 **Gabarito · o que observar**
>
> - Validação cruzada: **MAE = 0,74 ± 0,06**. Teste: **MAE = 0,71** e **R² = 0,50**.
> - Melhor que a reta só com horas (MAE 0,79 e R² 0,47): as outras features ajudam.
> - O teste ficou dentro da faixa prevista pela validação cruzada. É isso que dá confiança de que o modelo generaliza.
> - Atenção: os testes das seções 3.2 e 3.3 não são exatamente os mesmos alunos (aqui os faltantes não foram removidos), então a comparação é aproximada.

In [ ]:
coefs = pd.Series(multipla.named_steps['regressao'].coef_,
                  index=multipla.named_steps['preparo'].get_feature_names_out()).round(2)
coefs.sort_values()

> 🔎 **Gabarito · o que observar**
>
> - Maior peso: **`horas_estudo` (+0,84)**, depois **`faltas` (−0,39)** e **`turno_noite` (−0,36)**.
> - `idade` (+0,07) e `renda_familiar` (−0,07) ficam perto de zero, como a correlação do Lab 00 indicava.
> - `monitoria_sim` (+0,27) e `monitoria_não` (−0,27) são **espelhadas**: as duas colunas carregam a mesma informação. A diferença entre elas (0,54) é o efeito estimado da monitoria.
> - Como as numéricas foram **padronizadas**, os coeficientes são comparáveis: +0,84 significa +0,84 ponto por **desvio-padrão** de horas (≈ 2,2 h), e não por hora.

**Para pensar:** quais coeficientes são maiores em módulo? Isso bate com a correlação que você viu no Lab 00? Coeficiente prova causa?

> 🔎 **Gabarito · o que observar**
>
> **Resposta esperada:** horas, faltas e turno da noite; isso bate com a correlação (+0,62 e −0,31) e com as médias por turno do Lab 00.
> Coeficiente **não prova causa**: ele mede associação nos dados. Só um experimento (ex.: sortear quem recebe monitoria) mostraria o efeito causal.

## 4 · A taxa de aprendizado
Rode a descida do gradiente com taxas diferentes e compare as curvas de perda.

In [ ]:
# TODO: compare 4 taxas: 0.001 (pequena), 0.01 (boa), 0.023 (quase no limite) e 0.03 (grande demais)
for alpha in [0.001, 0.01, 0.023, 0.03]:
    _, _, h = descida_gradiente(x_tr, y_tr, alpha=alpha, epocas=300)
    plt.plot(np.clip(h, None, 1e3), label=f'α = {alpha}')
plt.yscale('log'); plt.xlabel('época'); plt.ylabel('MSE (treino)')
plt.legend(); plt.show()

> 🔎 **Gabarito · o que observar**
>
> Valores depois das 300 épocas (a reta “certa” é w ≈ 0,381 e b ≈ 1,772):
>
> | α | w | b | o que acontece |
> |---|---|---|---|
> | 0,001 | 0,61 | 0,20 | **lento**: a MSE mal saiu do lugar e `b` quase não foi aprendido |
> | 0,01 | 0,51 | 0,93 | desce bem, mas ainda **não convergiu** em 300 épocas |
> | 0,023 | 0,43 | 1,43 | a mais rápida, quase no limite |
> | 0,03 | ≈ −10⁵³ | ≈ −10⁵² | **diverge**: a MSE explode (o gráfico corta em 1 000) |
>
> - Para estes dados, o limite teórico é α ≈ 0,024. Acima dele, cada passo **ultrapassa** o mínimo por mais do que estava antes, e o erro cresce a cada época.
> - Com α = 0,023, o `w` **oscila**: 1,21 → 0,09 → 1,12 → 0,17 → … pulando de um lado para o outro do mínimo, enquanto a MSE ainda cai. A oscilação está nos parâmetros, não na curva.

**Para pensar:** qual curva desce devagar? Qual oscila? Qual explode (diverge)?
O que acontece com `w` quando a taxa é grande demais?

> 🔎 **Gabarito · o que observar**
>
> **Resposta esperada:** α = 0,001 desce devagar; α = 0,023 oscila nos parâmetros (vai e volta em torno do mínimo); α = 0,03 diverge.
> Com taxa grande demais, o `w` troca de sinal e cresce em módulo a cada época, até números astronômicos.

## 5 · Desafio extra: regularização
Um polinômio de grau alto em `horas_estudo` pode decorar o treino. A regressão **Ridge** soma à perda a penalidade
$\lambda \sum w^2$ (no Scikit-learn, $\lambda$ se chama `alpha`). Compare pela validação cruzada.

In [ ]:
def poli(lam):
    passos = [('poli', PolynomialFeatures(degree=10)), ('padronizar', StandardScaler())]
    passos.append(('reg', LinearRegression() if lam == 0 else Ridge(alpha=lam)))
    return Pipeline(passos)

for lam in [0, 0.1, 10, 1000]:
    # TODO: MAE médio da validação cruzada (cv=5) para cada λ
    mae = -cross_val_score(poli(lam), x_tr.reshape(-1, 1), y_tr, cv=5,
                           scoring='neg_mean_absolute_error').mean()
    print(f'λ = {lam:>6}: MAE = {mae:.3f}')

> 🔎 **Gabarito · o que observar**
>
> | λ | MAE (validação cruzada) | leitura |
> |---|---|---|
> | 0 | **2,60** | polinômio de grau 10 sem freio: decora o treino e erra feio em dados novos |
> | 0,1 | **0,825** | melhor: a penalidade “alisa” a curva |
> | 10 | 0,845 | ainda bom |
> | 1000 | 0,992 | freio demais: a curva fica quase reta e horizontal (subajuste) |
>
> - Nenhum polinômio regularizado bate a **reta simples** (MAE ≈ 0,79): a relação real é linear, e complexidade extra não paga.

## Para discutir
1. O modelo aprende com o **número de épocas** ou com a **atualização guiada pelo erro**?
2. O MAE de ~0,7 ponto é bom ou ruim? Depende de quê?
3. Por que a regressão múltipla erra menos que a reta só com horas?

> 🔎 **Gabarito · o que observar**
>
> **Pontos para a discussão:**
> 1. Com a atualização guiada pelo erro. Épocas são só repetições: se o passo for ruim, mais épocas não ajudam (α = 0,03).
> 2. Depende da escala e do uso. Numa nota de 0 a 10, 0,7 ponto é razoável para orientar um aluno, mas não para decidir aprovação perto do 6.
> 3. Porque faltas, turno e monitoria explicam parte da nota que as horas não explicam. O R² sobe de 0,47 para 0,50.